# Personal Protective Equipment Detection using YOLOv9c


This notebook demonstrates how to detect Personal Protective Equipment (PPE) such as ['Boots', 'Ear-protection', 'Glass', 'Glove', 'Helmet', 'Mask', 'Person', 'Vest'] using the YOLOv9c object detection model.

## Step 1: Install Required Packages
We begin by installing the [Ultralytics](https://github.com/ultralytics/ultralytics) YOLOv9c library.

In [ ]:
!pip install ultralytics

## Run Environment Checks
We verify if the YOLO library and dependencies are properly installed.

In [ ]:
import ultralytics
ultralytics.checks()

## Step 2: Import Required Modules for YOLOv9c and Visualization


In [ ]:
from ultralytics import YOLO
from IPython.display import Image

## Step 3: Download PPE Dataset from Roboflow

In this step, we use the [Roboflow](https://roboflow.com) Python API to download a custom PPE (Personal Protective Equipment) dataset that has been pre-annotated for object detection.

The following tasks are performed:
- Install the Roboflow Python client
- Authenticate using your Roboflow API key
- Access a specific project and version hosted under the workspace `badminton-szmsf`
- Download the dataset in the **YOLOv9c format** for training



In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="PqF6HU4FUT9iaWwYC0Yu")
project = rf.workspace("badminton-szmsf").project("safety-pyazl-6lzii")
version = project.version(1)
dataset = version.download("yolov9")


In [ ]:
dataset.location

- change the paths of image folder in .yaml file

## Step 4: Model Training.


In [ ]:
!yolo task = detect mode = train data = {dataset.location}/data.yaml model = "yolov9c.pt" epochs = 80 imgsz = 640

## Step 5: Model Evaluation.


In [ ]:
Image(r"/content/runs/detect/train/confusion_matrix.png", width= 800)

In [ ]:
Image(r"/content/runs/detect/train-3/labels.jpg", width=600)

In [ ]:
Image("/content/runs/detect/train/results.png", width=1000)

In [ ]:
!yolo task=detect mode=val model="/content/runs/detect/train/weights/best.pt" data="/content/safety-1/data.yaml"

In [ ]:
!yolo task=detect mode=predict model="/content/runs/detect/train/weights/best.pt" conf=0.25 source="/content/safety-1/test/images" save=True

## Step 6: Display Prediction Results

After running inference, YOLOv9c saves the prediction images (with bounding boxes) in the `runs/detect/predict/` folder.

In this step, we:
- Find the latest prediction folder
- Load the top few `.jpg` prediction images
- Display them inline to visualize the results


In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

latest_folder = max(glob.glob("/content/runs/detect/predict/"), key=os.path.getmtime)
for img in glob.glob("f{latest_folder}/*.jpg")[1:4]:
  display(IPyImage(filename=img, width=600))
  print("/n")



## Step 7: Run Inference using YOLOv9c Model

In this step, we use the trained YOLOv9c model (`best.pt`) to detect objects on a test image (`0.jpg`).  
The command sets a confidence threshold of **0.25**, and saves the output with detected bounding boxes.

- **Model**: `/content/runs/detect/train/weights/best.pt`  
- **Image Source**: `/content/0.jpg`  
- **Confidence Threshold**: `0.25`  
- **Output Folder**: Automatically saved in `runs/detect/predict/`


Visualize YOLOv9c Prediction Output

In [ ]:
!yolo task=detect mode=predict model="/content/runs/detect/train/weights/best.pt" conf=0.25 source="/content/coc.jpg" save=True

In [ ]:
Image("/content/runs/detect/predict-2/coc.jpg", width=600)

## Step 7 Run Inference on a Video 

In this step, we use the trained YOLOv9c model to perform object detection on a video file (`videoplayback.avi`).  
The prediction results (bounding boxes and labels) will be rendered frame by frame and saved as a new video in the `runs/detect/predict/` directory.

- **Model**: `/content/runs/detect/train/weights/best.pt`  
- **Video Source**: `/content/videoplayback.avi`  
- **Confidence Threshold**: `0.25`  
- **Output**: Annotated video saved automatically


In [ ]:
!yolo task=detect mode=predict model="/content/runs/detect/train/weights/best.pt" source="/content/vid.mp4" conf=0.25 save=True

## Step 8: Compress and Display YOLOv9c Inference Video

After running inference on a video file, YOLOv9c saves the output in `.avi` format, which may not play smoothly in web interfaces like Jupyter/Colab.

In this step, we:
- Convert the `.avi` output video to `.mp4` format using **FFmpeg** for better compatibility
- Encode the `.mp4` as Base64
- Embed and display the video directly within the notebook using HTML

This provides a smooth, inline preview of detection results on the full video.


In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

save_path = "/content/runs/detect/predict-3/vid.avi"

compress_path = "/content/vidplayback.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compress_path}")

mp4 = open(compress_path, "rb").read()

data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML("""
<video width=400 controls>
  <source src="%s" type="video/mp4">
</video>
""" % data_url)


In [ ]:
!pip install streamlit

## Step 9: Zip and Download YOLOv9c Results Folder via Streamlit

To simplify access to the detection results (images, videos, logs, etc.), we provide an option to **zip the entire `/runs` folder** and enable users to **download it directly** from the Streamlit app.

This step includes:
- Recursively zipping the contents of the `/runs` directory (which contains training and prediction outputs)
- Displaying a download button in the Streamlit UI for users to get the zipped file

Click the **"📦 Download Results Folder"** button to save your detection results locally.


In [ ]:
import streamlit as st
import zipfile
import os
from io import BytesIO

def zip_folder(folder_path):
    """Zips the contents of a folder and returns it as a BytesIO object"""
    zip_buffer = BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=folder_path)
                zip_file.write(file_path, arcname)
    zip_buffer.seek(0)
    return zip_buffer

# Example usage
folder_to_zip = "/content/runs"  # Replace with your folder path

if os.path.exists(folder_to_zip):
    zipped_folder = zip_folder(folder_to_zip)

    st.download_button(
        label="📦 Download Results Folder",
        data=zipped_folder,
        file_name="runs.zip",
        mime="application/zip"
    )
else:
    st.warning(f"Folder '{folder_to_zip}' not found.")


<hr>
